# Shoof layer model: photo data (step 1b)
Makes photo posters with exact layers: real free photos (CC0 / public domain from Openverse) as backgrounds, people and objects cut out of other photos, their shadows and rim lights, and text with glow / shadow / outline as separate layers. A third are real photos with their own subject layer.

**Settings:** Internet **On**, Accelerator **None**. Then **Save Version → Save & Run All**. About 2–3 hours. When it is done, add it as an input to step 2 (next to the step 1 notebook).

In [ ]:
N = 25000
!git clone -q --depth 1 https://github.com/shahrosekhalid96-debug/shoof /kaggle/temp/shoof
!pip install -q fonttools brotli onnxruntime
import os; os.chdir('/kaggle/temp/shoof'); print(os.cpu_count(), 'CPUs')

In [ ]:
# background photos and subject photos (free licences only; sources listed in SOURCES.tsv)
!python tools/layerdata/photos.py --out /kaggle/temp/bg --n 2500 --topics "city skyline night street neon room interior office studio stage concert sky sunset beach mountain forest desert wall texture bokeh lights space galaxy cafe kitchen market road bridge rain food restaurant table desk garden park snow river lake field farm museum church library gym classroom shop hotel airport train beach-house party wedding"
!python tools/layerdata/photos.py --out /kaggle/temp/subj --n 3000 --topics "person portrait man woman people model dancer athlete musician chef worker student robot dog cat bird horse car bicycle shoes bag bottle cup chair plant flower fruit toy statue guitar camera food plate cake pizza burger laptop phone watch lamp vase sofa table toy-car motorbike truck boat plane tree house sculpture painting book backpack hat sneakers"

In [ ]:
from PIL import Image
!python tools/layerdata/cutouts.py --photos /kaggle/temp/subj --out /kaggle/temp/cuts
# Shary's object sheet: clean cut-outs (x3 so they show up often) + his photo grid as backgrounds
import shutil, glob
for f in glob.glob('tools/layerdata/extra/cuts/*.png'):
    for k in range(3): shutil.copy(f, f'/kaggle/temp/cuts/x{k}-' + os.path.basename(f))
for f in glob.glob('tools/layerdata/extra/photos/*'):
    Image.open(f).convert('RGB').save('/kaggle/temp/bg/' + os.path.splitext(os.path.basename(f))[0] + '.jpg', quality=92)
from PIL import Image; import glob
cs = sorted(glob.glob('/kaggle/temp/cuts/*.png'))[:24]; S = Image.new('RGB', (6*200, 4*200), (40, 40, 40))
for i, f in enumerate(cs):
    im = Image.open(f).convert('RGBA'); im.thumbnail((200, 200)); b = Image.new('RGBA', im.size, (40, 40, 40, 255)); b.alpha_composite(im); S.paste(b.convert('RGB'), ((i % 6)*200, (i//6)*200))
S.save('/kaggle/working/cutouts-sheet.png'); S

In [ ]:
!python tools/layerdata/photo_gen.py --bgs /kaggle/temp/bg --cuts /kaggle/temp/cuts --subj /kaggle/temp/subj --out /kaggle/temp/lp --n {N} --start 1000000 --workers {os.cpu_count()} --sheet /kaggle/working/photo-sheet.png
from IPython.display import Image as I; display(I('/kaggle/working/photo-sheet.png'))

In [ ]:
import tarfile, shutil
ds = sorted(os.listdir('/kaggle/temp/lp')); print(len(ds), 'photo posters')
for k in range(0, len(ds), 5000):
    with tarfile.open(f'/kaggle/working/lp-{k//5000:02d}.tar', 'w') as t:
        for d in ds[k:k+5000]: t.add(f'/kaggle/temp/lp/{d}', arcname=d)
for f in ('bg', 'subj'): shutil.copy(f'/kaggle/temp/{f}/SOURCES.tsv', f'/kaggle/working/sources-{f}.tsv')
print(os.listdir('/kaggle/working'))